# Descripcion del conjunto de datos

Este notebook resume las variables, tipos y la identificacion del problema para `Datos.csv`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('Datos.csv')
print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])
df.head()

Filas: 683
Columnas: 11


,Sample code number,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1,3,1,1,2
1,1002945,5,4,4,5,7,10,3,2,1,2
2,1015425,3,1,1,1,2,2,3,1,1,2
3,1016277,6,8,8,1,3,4,3,7,1,2
4,1017023,4,1,1,3,2,1,3,1,1,2


In [2]:
descripcion_variables = pd.DataFrame({
    'variable': [
        'Sample code number',
        'Clump Thickness',
        'Uniformity of Cell Size',
        'Uniformity of Cell Shape',
        'Marginal Adhesion',
        'Single Epithelial Cell Size',
        'Bare Nuclei',
        'Bland Chromatin',
        'Normal Nucleoli',
        'Mitoses',
        'Class'
    ],
    'descripcion': [
        'Identificador de la muestra (no predictor clinico).',
        'Grosor del agrupamiento celular (1-10).',
        'Uniformidad del tamano celular (1-10).',
        'Uniformidad de la forma celular (1-10).',
        'Adhesion marginal celular (1-10).',
        'Tamano de celula epitelial individual (1-10).',
        'Nucleos desnudos (1-10).',
        'Cromatina blanda (1-10).',
        'Nucleolos normales (1-10).',
        'Mitosis observadas (1-10).',
        'Clase diagnostica: 2 = benigno, 4 = maligno.'
    ]
})
descripcion_variables

,variable,descripcion
0,Sample code number,Identificador de la muestra (no predictor clin...
1,Clump Thickness,Grosor del agrupamiento celular (1-10).
2,Uniformity of Cell Size,Uniformidad del tamano celular (1-10).
3,Uniformity of Cell Shape,Uniformidad de la forma celular (1-10).
4,Marginal Adhesion,Adhesion marginal celular (1-10).
5,Single Epithelial Cell Size,Tamano de celula epitelial individual (1-10).
6,Bare Nuclei,Nucleos desnudos (1-10).
7,Bland Chromatin,Cromatina blanda (1-10).
8,Normal Nucleoli,Nucleolos normales (1-10).
9,Mitoses,Mitosis observadas (1-10).


In [3]:
perfil = pd.DataFrame({
    'tipo_pandas': df.dtypes.astype(str),
    'faltantes': df.isna().sum(),
    'unicos': df.nunique()
}).reset_index().rename(columns={'index': 'variable'})

perfil['tipo_inferido'] = np.where(
    perfil['variable'].eq('Sample code number'),
    'id',
    np.where(
        perfil['variable'].eq('Class'),
        'objetivo_categorica_binaria',
        'predictora_numerica_ordinal'
    )
)
perfil

,variable,tipo_pandas,faltantes,unicos,tipo_inferido
0,Sample code number,int64,0,630,id
1,Clump Thickness,int64,0,10,predictora_numerica_ordinal
2,Uniformity of Cell Size,int64,0,10,predictora_numerica_ordinal
3,Uniformity of Cell Shape,int64,0,10,predictora_numerica_ordinal
4,Marginal Adhesion,int64,0,10,predictora_numerica_ordinal
5,Single Epithelial Cell Size,int64,0,10,predictora_numerica_ordinal
6,Bare Nuclei,int64,0,10,predictora_numerica_ordinal
7,Bland Chromatin,int64,0,10,predictora_numerica_ordinal
8,Normal Nucleoli,int64,0,10,predictora_numerica_ordinal
9,Mitoses,int64,0,9,predictora_numerica_ordinal


## Identificacion del problema

- Tipo de problema: **Clasificacion binaria**.
- Variable objetivo: **Class** (2 = benigno, 4 = maligno).
- Variables predictoras: 9 variables citologicas en escala 1-10.
- Columna a excluir para modelado: `Sample code number` (identificador).

In [4]:
dist_class = df['Class'].value_counts().sort_index().rename_axis('Class').reset_index(name='conteo')
dist_class['porcentaje'] = (dist_class['conteo'] / len(df) * 100).round(2)
dist_class

,Class,conteo,porcentaje
0,2,444,65.01
1,4,239,34.99


## Hipotesis iniciales

1. Mayores valores en `Uniformity of Cell Size`, `Uniformity of Cell Shape` y `Bare Nuclei` se asocian con clase maligna (4).
2. Existe correlacion positiva entre varias variables citologicas, por lo que algunas pueden ser redundantes.
3. El conjunto esta moderadamente desbalanceado hacia clase benigna, lo que sugiere monitorear metricas como recall y F1 para la clase maligna.